# Mean-Reversion vs Momentum — Regime-Aware Strategy

**Architecture (no DBTS, fixed config targets):**

**Layer 1 — Two ElasticNet regressors per target** (refit every 50 days,
expanding window, exponential time-decay weights α=0.99):
  - *Price regressor*: predicts `p_target_{t+1}` from 10 peers' prices today.
    → `price_residual = actual - predicted`, `residual_z = residual / σ_rolling`
  - *Return regressor*: predicts `r_target_{t+1}` from 10 peers' returns today.
    → `predicted_return`

**Layer 1 (parallel) — Regime classifier**
Input: ~15 binary technical-rule features (MA crosses, RSI bands, BB breaches,
breakouts, vol spikes, candle patterns).
Output: regime ∈ {−1=MEAN_REV, 0=NEUTRAL, +1=MOMENTUM}.

Label rule (price-only, no leakage of residual):
```
  prev_h_ret = p_t / p_{t-h} - 1
  next_h_ret = p_{t+h} / p_t - 1
  vol_thr    = σ_daily * sqrt(h)
  if |next_h_ret| < vol_thr            → NEUTRAL  (0)
  elif sign(next) == sign(prev) and |prev| > vol_thr → MOMENTUM (+1)
  elif sign(next) != sign(prev) and |prev| > vol_thr → MEAN_REVERSION (-1)
  else                                  → NEUTRAL  (0)
```

**Layer 2 — PositionManager (rule-based)** turns `regime + residual_z + predicted_return`
into LONG/SHORT/FLAT. The classifier never predicts direction.

**Targets**: fixed per sector from `config.SECTORS` (FCX, META, XOM, JPM, NVDA, PG,
PLD, NEE, UNH, AMZN). One position per sector at a time.

**Cache flag** `FORCE_RECOMPUTE` controls rebuilds; cache lives in
`outputs/regime_cache/`.

In [ ]:
# Cell 2 — Imports, cache flag, RNG seeds
import os, sys, json, math, pickle, warnings, hashlib
import random as _random
from pathlib import Path
from datetime import datetime
from itertools import product

import numpy as np
import pandas as pd
from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
%matplotlib inline

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'strategy').exists():
    cand = Path(r'C:\algo-trading-project')
    if cand.exists():
        PROJECT_ROOT = cand
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

# ===== CACHE FLAG (master switch) =====
FORCE_RECOMPUTE = False
# Per-stage overrides (default to FORCE_RECOMPUTE).
RECOMPUTE_REGRESSORS    = FORCE_RECOMPUTE
RECOMPUTE_TECH_FEATURES = FORCE_RECOMPUTE
RECOMPUTE_CLASSIFIER    = FORCE_RECOMPUTE
RECOMPUTE_PM            = True    # cheap, always rerun
# =======================================

CACHE_DIR = Path('outputs/regime_cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
_random.seed(SEED)
np.random.seed(SEED)

print(f'Project root    : {PROJECT_ROOT}')
print(f'FORCE_RECOMPUTE : {FORCE_RECOMPUTE}')
print(f'Cache dir       : {CACHE_DIR.resolve()}')
print(f'Seed            : {SEED}')

In [ ]:
# Cell 3 — Project imports, config, splits, data
from config import SECTORS
from strategy.strategy_config import StrategyConfig
from strategy.pipeline import StrategyPipeline
from strategy.splits import chrono_split

cfg = StrategyConfig(force_recompute=False, make_plots=False)
pipeline = StrategyPipeline(cfg)

print('Loading market data...')
md = pipeline.load_data()
split = chrono_split(md.prices.index, cfg)

train_idx = pd.DatetimeIndex(split.train_idx).sort_values()
val_idx   = pd.DatetimeIndex(split.val_idx).sort_values()
test_idx  = pd.DatetimeIndex(split.test_idx).sort_values()

full_idx = md.prices.index.sort_values()

TARGETS = {s['name']: {'etf': etf, 'target': s['target'], 'predictors': s['predictors']}
           for etf, s in SECTORS.items()}
print('Fixed targets per sector:')
for sector, info in TARGETS.items():
    print(f"  {sector:18s} | target={info['target']:5s} | predictors={info['predictors']}")

print(f'\nTRAIN : {train_idx[0].date()} -> {train_idx[-1].date()} | n={len(train_idx)}')
print(f'VAL   : {val_idx[0].date()}   -> {val_idx[-1].date()}   | n={len(val_idx)}')
print(f'TEST  : {test_idx[0].date()}  -> {test_idx[-1].date()}  | n={len(test_idx)} (LOCKED)')

In [ ]:
# Cell 4 — Helpers: technical features, regime label, walk-forward ElasticNet
RETRAIN_EVERY = 50
MIN_TRAIN_DAYS = 50
DECAY_ALPHA   = 0.995  # half-life ~138 days
EN_ALPHA      = 0.01
EN_L1_RATIO   = 0.5
EN_MAX_ITER   = 5000
RESID_Z_WIN   = 60     # rolling std window for residual z-score

LABEL_H       = 5      # horizon for prev/next h-day return
VOL_WIN       = 20     # rolling daily-return std window

def decay_weights(n, alpha=DECAY_ALPHA):
    """Exponential decay: most recent point weight=1, oldest = alpha^(n-1)."""
    return alpha ** np.arange(n-1, -1, -1)

def make_regime_label(close: pd.Series, h: int = LABEL_H, vol_win: int = VOL_WIN) -> pd.Series:
    """Trinary regime label: -1 MEAN_REV, 0 NEUTRAL, +1 MOMENTUM."""
    ret_daily = close.pct_change(fill_method=None)
    sigma     = ret_daily.rolling(vol_win, min_periods=10).std()
    vol_thr   = sigma * math.sqrt(h)
    prev = close / close.shift(h) - 1.0
    nxt  = close.shift(-h) / close - 1.0
    lab = pd.Series(0, index=close.index, dtype=float)
    small = nxt.abs() < vol_thr
    mom   = (~small) & (np.sign(nxt) == np.sign(prev)) & (prev.abs() > vol_thr)
    mr    = (~small) & (np.sign(nxt) != np.sign(prev)) & (prev.abs() > vol_thr)
    lab[mom] =  1
    lab[mr]  = -1
    lab[nxt.isna() | prev.isna() | sigma.isna()] = np.nan
    return lab.rename('regime_label')

def make_tech_features(close: pd.Series) -> pd.DataFrame:
    """~15 binary technical-rule features. Strictly causal (no shift(-))."""
    f = pd.DataFrame(index=close.index)
    ma20  = close.rolling(20,  min_periods=10).mean()
    ma50  = close.rolling(50,  min_periods=25).mean()
    ma200 = close.rolling(200, min_periods=100).mean()
    std20 = close.rolling(20,  min_periods=10).std()
    ret   = close.pct_change(fill_method=None)
    vol20 = ret.rolling(20, min_periods=10).std()
    # RSI(14)
    delta = close.diff()
    gain  = delta.where(delta > 0, 0.0).rolling(14, min_periods=7).mean()
    loss  = (-delta.where(delta < 0, 0.0)).rolling(14, min_periods=7).mean()
    rs    = gain / loss.replace(0, np.nan)
    rsi14 = 100 - 100 / (1 + rs)
    # Rolling max/min
    rmax20 = close.rolling(20, min_periods=10).max()
    rmin20 = close.rolling(20, min_periods=10).min()
    # Range
    rng_today  = (close - close.shift(1)).abs()
    rng_rmean  = rng_today.rolling(20, min_periods=10).mean()

    f['ma_short_above_long'] = (ma20  > ma50).astype(float)
    f['ma_med_above_long']   = (ma50  > ma200).astype(float)
    f['golden_cross_recent'] = ((ma50 > ma200) & (ma50.shift(20) < ma200.shift(20))).astype(float)
    f['death_cross_recent']  = ((ma50 < ma200) & (ma50.shift(20) > ma200.shift(20))).astype(float)
    f['price_above_ma50']    = (close > ma50).astype(float)
    f['price_above_ma200']   = (close > ma200).astype(float)
    f['rsi_overbought']      = (rsi14 > 70).astype(float)
    f['rsi_oversold']        = (rsi14 < 30).astype(float)
    f['bb_upper_breach']     = (close > (ma20 + 2*std20)).astype(float)
    f['bb_lower_breach']     = (close < (ma20 - 2*std20)).astype(float)
    f['breakout_20d_high']   = (close >= rmax20).astype(float)
    f['breakdown_20d_low']   = (close <= rmin20).astype(float)
    f['vol_spike']           = (ret.abs() > 2*vol20).astype(float)
    f['consec_up_3d']        = ((ret > 0) & (ret.shift(1) > 0) & (ret.shift(2) > 0)).astype(float)
    f['consec_down_3d']      = ((ret < 0) & (ret.shift(1) < 0) & (ret.shift(2) < 0)).astype(float)
    f['narrow_range']        = (rng_today < 0.5*rng_rmean).astype(float)
    return f.fillna(0.0)

def walk_forward_elasticnet(X: pd.DataFrame, y: pd.Series,
                             retrain_every: int = RETRAIN_EVERY,
                             min_train: int = MIN_TRAIN_DAYS,
                             alpha_decay: float = DECAY_ALPHA,
                             en_alpha: float = EN_ALPHA,
                             en_l1: float = EN_L1_RATIO):
    """Expanding-window walk-forward ElasticNet with exp-decay sample weights.

    Returns pd.Series of out-of-sample predictions aligned on X.index. Indices
    before `min_train` are NaN.
    """
    common = X.dropna().index.intersection(y.dropna().index)
    X, y = X.loc[common], y.loc[common]
    n = len(X)
    preds = pd.Series(np.nan, index=X.index)
    if n <= min_train:
        return preds
    start = min_train
    while start < n:
        stop = min(start + retrain_every, n)
        X_tr, y_tr = X.iloc[:start], y.iloc[:start]
        w_tr = decay_weights(len(y_tr), alpha_decay)
        sc = StandardScaler().fit(X_tr.values)
        X_tr_s = sc.transform(X_tr.values)
        model = ElasticNet(alpha=en_alpha, l1_ratio=en_l1,
                           max_iter=EN_MAX_ITER, random_state=SEED, fit_intercept=True)
        model.fit(X_tr_s, y_tr.values, sample_weight=w_tr)
        X_pr_s = sc.transform(X.iloc[start:stop].values)
        preds.iloc[start:stop] = model.predict(X_pr_s)
        start = stop
    return preds

print('Helpers ready.')
print(f'  Regressor refit: every {RETRAIN_EVERY}d, min_train={MIN_TRAIN_DAYS}, decay α={DECAY_ALPHA}')
print(f'  Regime label   : h={LABEL_H}, vol_win={VOL_WIN}')

In [ ]:
# Cell 5 — Build (or load) walk-forward regressor outputs per target
REGRESSOR_CACHE = CACHE_DIR / 'regressor_outputs.pkl'

if REGRESSOR_CACHE.exists() and not RECOMPUTE_REGRESSORS:
    print(f'[CACHE HIT] {REGRESSOR_CACHE.name}')
    with open(REGRESSOR_CACHE, 'rb') as f:
        regressor_outputs = pickle.load(f)
else:
    print('[RECOMPUTE] fitting WF ElasticNet (price + return) per target...')
    regressor_outputs = {}
    for sector, info in TARGETS.items():
        tgt, preds = info['target'], info['predictors']
        peers = [p for p in preds if p in md.prices.columns]
        if tgt not in md.prices.columns or not peers:
            print(f'  SKIP {sector}: missing data')
            continue
        # === PRICE regressor: CONTEMPORANEOUS — peers' price today → target price today ===
        # Cross-sectional spread: residual_z captures how 'over/under-valued' the
        # target is vs its peer-implied price RIGHT NOW. This is the canonical
        # pairs-trading signal.
        X_price = md.prices[peers].copy()
        y_price = md.prices[tgt].copy()        # contemporaneous, not shifted
        pred_price = walk_forward_elasticnet(X_price, y_price)
        residual   = md.prices[tgt] - pred_price
        residual_z = (residual - residual.rolling(RESID_Z_WIN, min_periods=20).mean()) / \
                      residual.rolling(RESID_Z_WIN, min_periods=20).std()

        # === RETURN regressor: peers' returns today → target's return next day ===
        X_ret = md.returns[peers].copy()
        y_ret = md.returns[tgt].shift(-1)
        pred_ret_raw = walk_forward_elasticnet(X_ret, y_ret)
        # pred_ret at index t = forecast of return at t+1 made from t's peer returns.
        # For PM use we want the forecast available at date t for trading at t (decision based on data <= t).
        # So predicted_return_for_today_trade = pred_ret_raw.shift(1) but actually pred_ret_raw[t]
        # was made FROM data at t already (peers known at close t), so it IS the forecast for t+1.
        # Trading at close of t for next bar means we use pred_ret_raw[t] directly.
        predicted_return = pred_ret_raw

        # realised next return (for PM PnL and label sanity)
        next_ret = md.returns[tgt].shift(-1)

        regressor_outputs[sector] = {
            'target': tgt, 'peers': peers,
            'pred_price_next':  pred_price,
            'residual':         residual,
            'residual_z':       residual_z,
            'predicted_return': predicted_return,
            'next_ret':         next_ret,
        }
        valid_rz = residual_z.notna().sum()
        print(f'  {sector:18s} | target={tgt:5s} | peers={len(peers):2d} | '
              f'valid rz={valid_rz} | valid pred_ret={predicted_return.notna().sum()}')

    with open(REGRESSOR_CACHE, 'wb') as f:
        pickle.dump(regressor_outputs, f)
    print(f'  saved -> {REGRESSOR_CACHE.name}')

# Quick sanity
print(f'\nRegressor outputs for {len(regressor_outputs)} sectors.')
_demo = next(iter(regressor_outputs.values()))
_summary = pd.DataFrame({
    'residual_z': _demo['residual_z'].describe(),
    'predicted_return': _demo['predicted_return'].describe(),
    'next_ret': _demo['next_ret'].describe(),
}).round(4)
print(f'\n--- {_demo["target"]} regressor outputs (descriptive) ---')
display(_summary)

In [ ]:
# Cell 6 — Build (or load) technical-rule feature panel for all targets
TECH_CACHE = CACHE_DIR / 'tech_features.pkl'

if TECH_CACHE.exists() and not RECOMPUTE_TECH_FEATURES:
    print(f'[CACHE HIT] {TECH_CACHE.name}')
    tech_panel = pd.read_pickle(TECH_CACHE)
else:
    print('[RECOMPUTE] building technical-rule features per target...')
    rows = []
    for sector, info in TARGETS.items():
        tgt = info['target']
        if tgt not in md.prices.columns:
            continue
        feats = make_tech_features(md.prices[tgt])
        feats['date']   = feats.index
        feats['sector'] = sector
        feats['target'] = tgt
        rows.append(feats.reset_index(drop=True))
    tech_panel = pd.concat(rows, ignore_index=True)
    tech_panel.to_pickle(TECH_CACHE)
    print(f'  saved -> {TECH_CACHE.name}')

TECH_FEATURE_COLS = [c for c in tech_panel.columns if c not in ('date','sector','target')]
print(f'\nTech panel: {len(tech_panel):,} rows | {len(TECH_FEATURE_COLS)} features')
print(f'Features: {TECH_FEATURE_COLS}')

# Distribution of one feature for sanity
_one = tech_panel.groupby('target')[TECH_FEATURE_COLS].mean().round(3)
print('\nMean activation rate per feature per target:')
display(_one)

In [ ]:
# Cell 7 — Build regime labels per target; merge with tech panel; train classifier
CLF_CACHE = CACHE_DIR / 'regime_clf.pkl'

# Compute labels (cheap; always fresh)
label_rows = []
for sector, info in TARGETS.items():
    tgt = info['target']
    if tgt not in md.prices.columns:
        continue
    lab = make_regime_label(md.prices[tgt])
    label_rows.append(pd.DataFrame({
        'date':   lab.index,
        'sector': sector,
        'target': tgt,
        'regime_label': lab.values,
    }))
labels = pd.concat(label_rows, ignore_index=True)

# Merge tech features ↔ labels
panel_clf = tech_panel.merge(labels, on=['date','sector','target'], how='left')
panel_clf['date'] = pd.to_datetime(panel_clf['date'])

# Train/val split by date
train_mask = panel_clf['date'].isin(train_idx)
val_mask   = panel_clf['date'].isin(val_idx)
data_tr = panel_clf[train_mask & panel_clf['regime_label'].notna()].copy()
data_va = panel_clf[val_mask   & panel_clf['regime_label'].notna()].copy()

print(f'Train rows (label != NaN) : {len(data_tr):,}')
print(f'Val   rows (label != NaN) : {len(data_va):,}')
print('\nLabel distribution (TRAIN):')
display(data_tr['regime_label'].value_counts().rename(index={-1:'MEAN_REV',0:'NEUTRAL',1:'MOMENTUM'}))

_TO_IDX = {-1: 0, 0: 1, 1: 2}; _FROM_IDX = {0: -1, 1: 0, 2: 1}
X_tr = data_tr[TECH_FEATURE_COLS].astype(float)
y_tr = data_tr['regime_label'].map(_TO_IDX).astype(int)
X_va = data_va[TECH_FEATURE_COLS].astype(float)
y_va = data_va['regime_label'].map(_TO_IDX).astype(int)

if CLF_CACHE.exists() and not RECOMPUTE_CLASSIFIER:
    print(f'[CACHE HIT] {CLF_CACHE.name}')
    with open(CLF_CACHE, 'rb') as f:
        clf_blob = pickle.load(f)
    clf = clf_blob['model']
else:
    print('[RECOMPUTE] fitting regime classifier on TRAIN...')
    sw = compute_sample_weight(class_weight='balanced', y=y_tr)
    clf = XGBClassifier(
        n_estimators=300, learning_rate=0.03, max_depth=3,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, reg_alpha=0.1,
        num_class=3, objective='multi:softprob', verbosity=0,
        random_state=SEED, seed=SEED, n_jobs=1, tree_method='exact',
    )
    clf.fit(X_tr, y_tr, sample_weight=sw)
    with open(CLF_CACHE, 'wb') as f:
        pickle.dump({'model': clf, 'features': TECH_FEATURE_COLS}, f)
    print(f'  saved -> {CLF_CACHE.name}')

# Eval on val
pred_va = clf.predict(X_va)
print(f'\nVAL metrics:')
print(f'  accuracy : {accuracy_score(y_va, pred_va):.4f}')
print(f'  f1_macro : {f1_score(y_va, pred_va, average="macro"):.4f}')
print('\nVAL confusion matrix (rows=true, cols=pred, labels=[MR,N,MOM]):')
print(pd.DataFrame(confusion_matrix(y_va, pred_va, labels=[0,1,2]),
                   index=['MR','N','MOM'], columns=['MR','N','MOM']))

imp = pd.Series(clf.feature_importances_, index=TECH_FEATURE_COLS).sort_values(ascending=False)
print('\nFeature importances:')
display(imp.to_frame('importance').round(4))

In [ ]:
# Cell 8 — Assemble decision panel: regime predictions + residual_z + predicted_return
# This panel feeds the PM. One row per (date, sector).

# Get clf predictions over ALL dates (train+val+test) so we can choose what to backtest on.
X_all = panel_clf[TECH_FEATURE_COLS].astype(float)
proba_all = clf.predict_proba(X_all)
pred_all  = np.array([_FROM_IDX[i] for i in clf.predict(X_all)])
panel_clf['regime_pred'] = pred_all
panel_clf['P_MR']        = proba_all[:, 0]
panel_clf['P_NEUTRAL']   = proba_all[:, 1]
panel_clf['P_MOM']       = proba_all[:, 2]
panel_clf['regime_conf'] = proba_all.max(axis=1)

decision_rows = []
for sector, info in TARGETS.items():
    tgt = info['target']
    if sector not in regressor_outputs:
        continue
    reg = regressor_outputs[sector]
    clf_slice = panel_clf[panel_clf['sector'] == sector][
        ['date','regime_pred','regime_conf','P_MR','P_NEUTRAL','P_MOM','regime_label']
    ].set_index('date')
    df = pd.DataFrame({
        'residual_z':       reg['residual_z'],
        'predicted_return': reg['predicted_return'],
        'next_ret':         reg['next_ret'],
        'price':            md.prices[tgt],
    })
    df = df.join(clf_slice, how='left')
    df['sector'] = sector
    df['target'] = tgt
    df['date']   = df.index
    decision_rows.append(df.reset_index(drop=True))

decision_panel = pd.concat(decision_rows, ignore_index=True)
decision_panel = decision_panel.dropna(subset=['residual_z','regime_pred','next_ret']).reset_index(drop=True)
decision_panel = decision_panel.sort_values(['date','sector']).reset_index(drop=True)
print(f'Decision panel: {len(decision_panel):,} rows ({decision_panel["sector"].nunique()} sectors, '
      f'{decision_panel["date"].nunique()} dates)')

print('\nRegime distribution across decision panel:')
display(decision_panel['regime_pred'].map({-1:'MEAN_REV',0:'NEUTRAL',1:'MOMENTUM'}).value_counts())

print('\nResidual_z descriptive:')
display(decision_panel['residual_z'].describe().round(3).to_frame('residual_z'))

In [ ]:
# Cell 9 — Regime-aware rule-based PositionManager + backtest
PM_COST_BPS = 5.0 / 1e4

# Gate parameters
RZ_THRESHOLD       = 1.5     # |residual_z| required for entry
REGIME_CONF_THR    = 0.55    # min regime probability
MR_EXIT            = 0.30    # exit when |rz| < this
MAX_HOLDING_DAYS   = 10
STOP_LOSS          = -0.02
TAKE_PROFIT        =  0.03
PRED_RET_THR       = 0.001   # for MOMENTUM regime, require |pred_ret| > this in same direction
BACKTEST_ON        = 'train' # 'train' or 'val'

def regime_pm_simulate(panel, cost_bps=PM_COST_BPS,
                       rz_thr=RZ_THRESHOLD, conf_thr=REGIME_CONF_THR,
                       mr_exit=MR_EXIT, max_hold=MAX_HOLDING_DAYS,
                       stop_loss=STOP_LOSS, take_profit=TAKE_PROFIT,
                       pred_ret_thr=PRED_RET_THR):
    """One position per sector. Returns a long-form trades DataFrame."""
    # state: per sector
    state = {}   # sector -> dict(position, days_in, entry_price, trade_pnl, trade_id)
    seq   = {}   # sector -> next trade id
    rows  = []
    for (date, sector), grp in panel.groupby(['date','sector'], sort=True):
        r = grp.iloc[0]
        st = state.get(sector, dict(position=0, days_in=0, entry_price=np.nan,
                                    trade_pnl=0.0, trade_id=None))
        prev_pos = st['position']
        rz    = float(r['residual_z'])
        regm  = int(r['regime_pred'])
        conf  = float(r['regime_conf'])
        prr   = float(r['predicted_return']) if pd.notna(r['predicted_return']) else 0.0
        nr    = float(r['next_ret']) if pd.notna(r['next_ret']) else 0.0
        price = float(r['price']) if pd.notna(r['price']) else np.nan
        action = 'HOLD'
        desired = prev_pos

        # === EXIT logic first ===
        if prev_pos != 0:
            st['days_in'] += 1
            do_exit = False
            reason = ''
            if st['days_in'] >= max_hold:
                do_exit = True; reason = 'max_hold'
            elif abs(rz) <= mr_exit:
                do_exit = True; reason = 'rz_revert'
            elif st['trade_pnl'] <= stop_loss:
                do_exit = True; reason = 'stop_loss'
            elif st['trade_pnl'] >= take_profit:
                do_exit = True; reason = 'take_profit'
            if do_exit:
                desired = 0
                action = 'EXIT'

        # === ENTRY logic (only if currently flat after potential exit) ===
        if desired == 0 and abs(rz) >= rz_thr and regm != 0 and conf >= conf_thr:
            if regm == -1:   # MEAN_REVERSION
                # rz << 0 → price below shadow → revert up → LONG; rz >> 0 → revert down → SHORT
                if rz < 0:
                    desired = 1;  action = 'ENTER_LONG'
                else:
                    desired = -1; action = 'ENTER_SHORT'
            else:           # MOMENTUM (+1) — trend continuation: follow residual_z direction
                # rz > 0 → target rich vs peers; in momentum it keeps drifting up → LONG
                # rz < 0 → target cheap vs peers; in momentum it keeps drifting down → SHORT
                # Require predicted_return agreement as sanity check.
                if rz > 0 and prr > -pred_ret_thr:
                    desired = 1;  action = 'ENTER_LONG'
                elif rz < 0 and prr < pred_ret_thr:
                    desired = -1; action = 'ENTER_SHORT'
                # else stay flat (rz and pred_ret disagree)

        turnover = abs(desired - prev_pos)
        gross    = desired * nr
        net      = gross - turnover * cost_bps

        is_entry = action.startswith('ENTER')
        is_exit  = (action == 'EXIT')
        if is_entry:
            seq[sector] = seq.get(sector, 0) + 1
            st = dict(position=desired, days_in=1, entry_price=price,
                      trade_pnl=net, trade_id=seq[sector])
        elif is_exit:
            st['trade_pnl'] += net
            realised = st['trade_pnl']
            st = dict(position=0, days_in=0, entry_price=np.nan,
                      trade_pnl=0.0, trade_id=None)
        else:
            if desired != 0:
                st['trade_pnl'] += net

        state[sector] = st
        rows.append({
            'date': date, 'sector': sector, 'target': r['target'],
            'residual_z': rz, 'regime_pred': regm, 'regime_conf': conf,
            'predicted_return': prr, 'next_ret': nr,
            'action': action, 'position': float(desired), 'prev_pos': float(prev_pos),
            'turnover': float(turnover),
            'gross_pnl': float(gross), 'net_pnl': float(net),
            'is_entry': bool(is_entry), 'is_exit': bool(is_exit),
            'days_in_position': int(st['days_in']) if desired != 0 else 0,
        })
    return pd.DataFrame(rows)

# Choose backtest window
if BACKTEST_ON == 'train':
    bt_idx = train_idx
elif BACKTEST_ON == 'val':
    bt_idx = val_idx
else:
    raise ValueError(BACKTEST_ON)

bt_panel = decision_panel[decision_panel['date'].isin(bt_idx)].copy()
print(f'Backtest window: {BACKTEST_ON} | '
      f'{bt_panel["date"].min().date()} -> {bt_panel["date"].max().date()} | '
      f'{bt_panel["date"].nunique()} dates | {len(bt_panel):,} rows')

trades = regime_pm_simulate(bt_panel)
print(f'\nTrade rows: {len(trades):,}')
print(f'Action distribution:\n{trades["action"].value_counts()}')
print(f'Entries: {int(trades["is_entry"].sum())} | Exits: {int(trades["is_exit"].sum())}')

In [ ]:
# Cell 10 — Portfolio metrics + log to run_history
def portfolio_metrics(trades_df, periods_per_year=252):
    if trades_df.empty:
        return pd.Series(dtype=float)
    daily = trades_df.groupby('date')['net_pnl'].mean().sort_index()
    if daily.empty:
        return pd.Series(dtype=float)
    equity = (1.0 + daily).cumprod()
    cum = float(equity.iloc[-1] - 1.0)
    n = len(daily)
    ann_ret = float((1.0 + cum) ** (periods_per_year / max(n,1)) - 1.0)
    ann_vol = float(daily.std(ddof=1) * np.sqrt(periods_per_year)) if n > 1 else np.nan
    sharpe = ann_ret / ann_vol if ann_vol and np.isfinite(ann_vol) and ann_vol != 0 else np.nan
    dn = daily[daily < 0]
    dn_vol = float(dn.std(ddof=1) * np.sqrt(periods_per_year)) if len(dn) > 1 else np.nan
    sortino = ann_ret / dn_vol if dn_vol and np.isfinite(dn_vol) and dn_vol != 0 else np.nan
    peak = equity.cummax()
    max_dd = float((equity / peak - 1.0).min())
    entries = trades_df[trades_df['is_entry']]
    return pd.Series({
        'trading_days':          n,
        'total_entries':         int(len(entries)),
        'long_entries':          int((entries['position'] == 1).sum()),
        'short_entries':         int((entries['position'] == -1).sum()),
        'active_days':           int((trades_df['position'] != 0).sum()),
        'cumulative_return':     round(cum, 4),
        'annualized_return':     round(ann_ret, 4),
        'annualized_volatility': round(ann_vol, 4) if np.isfinite(ann_vol) else np.nan,
        'sharpe':                round(sharpe, 4) if np.isfinite(sharpe) else np.nan,
        'sortino':               round(sortino, 4) if np.isfinite(sortino) else np.nan,
        'max_drawdown':          round(max_dd, 4),
        'win_rate_days':         round(float((daily > 0).mean()), 4),
        'avg_daily_pnl':         round(float(daily.mean()), 6),
    })

metrics = portfolio_metrics(trades)
print('=' * 60)
print(f'REGIME-AWARE STRATEGY METRICS ({BACKTEST_ON.upper()})')
print('=' * 60)
print(metrics.to_string())

# === Append to run_history.csv ===
RUN_HISTORY = Path('outputs/tuning_cache/run_history.csv')
RUN_HISTORY.parent.mkdir(parents=True, exist_ok=True)
HIST_COLS = ['timestamp','source','tag','clf_trial','f1_val','rz_thr','conf','mr_exit',
             'total_entries','sharpe','sortino','cumulative_return',
             'annualized_return','annualized_volatility','max_drawdown',
             'win_rate_days','active_days']
row = {
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'source':    'Regime_MeanRev_vs_Mom',
    'tag':       f'rz{RZ_THRESHOLD}_conf{REGIME_CONF_THR}_mre{MR_EXIT}_{BACKTEST_ON}',
    'clf_trial': None, 'f1_val': None,
    'rz_thr': RZ_THRESHOLD, 'conf': REGIME_CONF_THR, 'mr_exit': MR_EXIT,
    **{k: metrics.get(k) for k in HIST_COLS if k not in ('timestamp','source','tag','clf_trial','f1_val','rz_thr','conf','mr_exit')},
}
new_df = pd.DataFrame([{c: row.get(c) for c in HIST_COLS}])
if RUN_HISTORY.exists():
    prev = pd.read_csv(RUN_HISTORY)
    combined = pd.concat([prev, new_df], ignore_index=True)
else:
    combined = new_df
combined.to_csv(RUN_HISTORY, index=False)
print(f'\nLogged to {RUN_HISTORY}')

# Per-sector breakdown
print('\nPer-sector metrics:')
sec_rows = []
for sec, g in trades.groupby('sector'):
    m = portfolio_metrics(g)
    sec_rows.append({
        'sector': sec,
        'entries': int(m.get('total_entries', 0)),
        'sharpe':  m.get('sharpe', np.nan),
        'sortino': m.get('sortino', np.nan),
        'cum_ret': m.get('cumulative_return', np.nan),
        'max_dd':  m.get('max_drawdown', np.nan),
        'win_rate_days': m.get('win_rate_days', np.nan),
    })
display(pd.DataFrame(sec_rows).sort_values('sharpe', ascending=False, na_position='last'))

In [ ]:
# Cell 11 — Show recent runs from run_history (for comparison)
if RUN_HISTORY.exists():
    hist = pd.read_csv(RUN_HISTORY)
    print(f'Total runs in history: {len(hist)} | sources: {hist["source"].value_counts().to_dict()}')
    print('\nTop 10 by Sharpe across ALL runs:')
    display(hist.sort_values('sharpe', ascending=False).head(10))
    print('\nMost recent 10 runs:')
    display(hist.tail(10))
else:
    print('No history file yet.')